## Парсинг данных с сайта на Python

### Условие задачи

С сайта ( https://lostfilm.info/top250/ ) необходимо построить исходный набор данных (.csv или .xml). Набор данных должен включать __название, жанр, год и страна, рейтинг, описание__. Подготовленный набор данных должен содержать сведения о 250 фильмах. Разработанный парсер должен извлекать гиперссылки из начальной страницы с последующим обходом всех страниц по полученным ссылкам и извлечением их содержимого. Можно дополнить набор какими-либо другими данными, если они могут быть полезны для дальнейшего исследования.


### Этапы парсинга

1. Поиск данных
2. Получение информации
3. Сохранение данных

### Подключение библиотек

In [24]:
from bs4 import BeautifulSoup as bs

Beautiful Soup - это библиотека Python для извлечения данных из HTML и XML файлов. 

In [25]:
import requests

Библиотека requests является стандартным инструментом для составления HTTP-запросов в Python.

In [26]:
import pandas as pd

### Получение информаций

In [32]:
url = 'https://lostfilm.info/top250/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7'
}

response = requests.get(url, headers=headers)
page = requests.get(url, headers=headers)
soup = bs(response.text, 'html.parser')


In [33]:
page.status_code

200

In [34]:
soup

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">

<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<title>250 лучших фильмов</title>
<link href="/favicon.ico" rel="icon" type="image/x-icon"/>
<link href="/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
<link href="/css/style_all.css?3" rel="stylesheet" type="text/css">
<style>body
{
	background: url('/2610d1f2ee8a953fb408bfc8fc4aad5a.jpg') top center no-repeat #addbe8;
}		</style><script language="javascript" src="https://www.gstatic.com/swiffy/v7.3.0/runtime.js?1" type="text/javascript"></script>
<script src="/js/js_all.js?1" type="text/javascript"></script>
<link href="/js/flowplayer/skin/functional.css" id="size-stylesheet" media="screen" rel="stylesheet" type="text/css">
<link href="/js/flowplayer/skin/quality.selector.css" id="size-stylesheet" media="screen" rel="stylesheet" type="text/css"/>
<

In [35]:
page.text

'<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">\r\n<html xmlns="http://www.w3.org/1999/xhtml">\r\n<head>\r\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\r\n<title>250 лучших фильмов</title>\r\n<link rel="icon" href="/favicon.ico" type="image/x-icon">\r\n<link rel="shortcut icon" href="/favicon.ico" type="image/x-icon"> \r\n<link type="text/css" href="/css/style_all.css?3" rel="stylesheet" />\r\n<style>body\n{\n\tbackground: url(\'/2610d1f2ee8a953fb408bfc8fc4aad5a.jpg\') top center no-repeat #addbe8;\n}\t\t</style><script src="https://www.gstatic.com/swiffy/v7.3.0/runtime.js?1" type="text/javascript" language="javascript"></script>\r\n<script type="text/javascript" src="/js/js_all.js?1"></script>\r\n\t\t<link id="size-stylesheet" rel="stylesheet" type="text/css" href="/js/flowplayer/skin/functional.css" media="screen" />\r\n\t\t<link id="size-stylesheet" rel="stylesheet" type="text/css" href="/js/flo

In [36]:
result_list = {
    'title': [],
    'title_eng': [],
    'genre': [],
    'year': [],
    'country': [],
    'rating': [],
    'description': []
}

### Алгоритм

In [37]:
import re
from time import sleep
import random
films_data = []
rows = soup.find_all('tr')[1:] 
for row in rows:
    cells = row.find_all('td')
    if len(cells) >= 4:
        title_cell = cells[1]
        link = title_cell.find('a')
        if link:
            film_url = "https://lostfilm.info" + link['href']
            title_rus = link.text.strip()
            small = title_cell.find('small')
            title_eng = small.text.strip() if small else ""
            year = cells[2].text.strip()
            rating = cells[3].text.strip()
            films_data.append({
                'title_rus': title_rus,
                'title_eng': title_eng,
                'year': year,
                'rating': rating,
                'url': film_url
            })
print(f"Найдено фильмов: {len(films_data)}")
for idx, film in enumerate(films_data[:250]):
    print(f"\n{film['title_rus']}")
    genre = "Н/Д"
    country = "Н/Д"
    description = "Н/Д"
    try:
        sleep(random.uniform(1, 2))
        film_response = requests.get(film['url'], timeout=10)
        if film_response.status_code == 200:
            film_soup = bs(film_response.text, 'html.parser')
            genre_span = film_soup.find('span', itemprop='genre')
            if genre_span:
                genre_links = genre_span.find_all('a')
                if genre_links:
                    genres = []
                    for link in genre_links:
                        genres.append(link.text.strip())
                    genre = ', '.join(genres)
                else:
                    genre = genre_span.text.strip()
            country_i = film_soup.find('i', class_='gray')
            if country_i:
                country_link = country_i.find_next('a', href=re.compile(r'/country/'))
                if country_link:
                    country = country_link.text.strip()
            story_div = film_soup.find('div', id='story')
            if story_div:
                description = story_div.get_text().strip()
            print(f"Жанр: {genre[:50]}")
            print(f"Страна: {country[:50]}")
            print(f"Описание: {'да' if description != 'Н/Д' else 'нет'}")
        else:
            print(f"Ошибка загрузки: {film_response.status_code}")
            
    except Exception as e:
        print(f"Ошибка: {e}")
    result_list['title'].append(film['title_rus'])
    result_list['title_eng'].append(film['title_eng'])
    result_list['genre'].append(genre)
    result_list['year'].append(film['year'])
    result_list['country'].append(country)
    result_list['rating'].append(film['rating'])
    result_list['description'].append(description)
    
    

Найдено фильмов: 250

Побег из Шоушенка
Жанр: Драма, Криминальный
Страна: США
Описание: да

Крестный отец
Жанр: Драма, Криминальный
Страна: США
Описание: да

Темный рыцарь
Жанр: Драма, Криминальный, Боевик, Триллер
Страна: США
Описание: да

Крестный отец 2
Жанр: Драма, Криминальный
Страна: США
Описание: да

Хороший, плохой, злой
Жанр: Вестерн, Приключения, Боевик
Страна: Испания
Описание: да

Криминальное чтиво
Жанр: Комедия, Драма, Криминальный, Триллер
Страна: США
Описание: да

Властелин колец: Возвращение короля
Жанр: Драма, Фэнтези, Приключения, Боевик
Страна: США
Описание: да

Список Шиндлера
Жанр: Драма, Военный, Биография, Исторический
Страна: США
Описание: да

Мир Дикого Запада
Жанр: Драма, Вестерн, Фантастика, Детектив
Страна: США
Описание: да

12 разгневанных мужчин
Жанр: Драма, Криминальный
Страна: США
Описание: да

Начало
Жанр: Драма, Криминальный, Фантастика, Приключения, Боев
Страна: США
Описание: да

Властелин колец: Братство Кольца
Жанр: Драма, Фэнтези, Приключения, Бое

In [38]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
title_eng - 0
genre - 0
year - 0
country - 0
rating - 0
description - 0


### Сохранение данных

In [39]:
df = pd.DataFrame(result_list)
df.to_csv('top250_final.csv', index=False, encoding='utf-8-sig')
print(f"\nфильмов: {len(result_list['title'])}")
print("Файл сохранен:top250_final.csv")


фильмов: 250
Файл сохранен:top250_final.csv


In [40]:
df.head(15)

,title,title_eng,genre,year,country,rating,description
0,Побег из Шоушенка,The Shawshank Redemption,"Драма, Криминальный",1994,США,9.20,Успешный банкир Энди Дюфрейн обвинен в убийств...
1,Крестный отец,The Godfather,"Драма, Криминальный",1972,США,9.20,"Криминальная сага, повествующая о нью-йоркской..."
2,Темный рыцарь,The Dark Knight,"Драма, Криминальный, Боевик, Триллер",2008,США,9.00,Бэтмен поднимает ставки в войне с криминалом. ...
3,Крестный отец 2,The Godfather: Part II,"Драма, Криминальный",1974,США,9.00,В центре драмы представители нового поколения ...
4,"Хороший, плохой, злой","Il buono, il brutto, il cattivo.","Вестерн, Приключения, Боевик",1966,Испания,9.00,В разгар гражданской войны таинственный стрело...
5,Криминальное чтиво,Pulp Fiction,"Комедия, Драма, Криминальный, Триллер",1994,США,8.90,Двое бандитов Винсент Вега и Джулс Винфилд про...
6,Властелин колец: Возвращение короля,The Lord of the Rings: The Return of the King,"Драма, Фэнтези, Приключения, Боевик",2003,США,8.90,"Заключительная часть кинотрилогии, снятой по р..."
7,Список Шиндлера,Schindler's List,"Драма, Военный, Биография, Исторический",1993,США,8.90,"7 номинаций на премию Оскар, включая премию за..."
8,Мир Дикого Запада,Westworld,"Драма, Вестерн, Фантастика, Детектив",2016,США,8.90,В футуристическом парке развлечений «Мир Диког...
9,12 разгневанных мужчин,12 Angry Men,"Драма, Криминальный",1957,США,8.90,"Юношу обвиняют в убийстве собственного отца, е..."


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        250 non-null    object
 1   title_eng    250 non-null    object
 2   genre        250 non-null    object
 3   year         250 non-null    object
 4   country      250 non-null    object
 5   rating       250 non-null    object
 6   description  250 non-null    object
dtypes: object(7)
memory usage: 13.8+ KB


### Работа с API

In [42]:
import json
API_TOKEN = "V6FCA5R-JSG4X38-N4EB7Q2-FGCV21D"
HEADERS = {'X-API-KEY': API_TOKEN}
response = requests.get(
    "https://api.poiskkino.dev/v1.4/movie",
    headers=HEADERS,
    params={'limit': 1}
)
if response.status_code == 200:
    data = response.json()
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    print(f"Ошибка {response.status_code}: {response.text}")

{
  "docs": [
    {
      "id": 11385598,
      "alternativeName": "Yйлену оңай",
      "countries": [
        {
          "name": "Казахстан"
        }
      ],
      "description": "Начинающий сценарист Айдар в поисках идеи для фильма об отношениях знакомится с психологом. Вместе они пытаются понять, что на самом деле происходит с любовью после свадьбы — и почему чувства, начинавшиеся как «навсегда», сталкиваются с реальностью. В их поле зрения — несколько историй. Успешный менеджер Жанар и её супруг Жандос, взявший на себя роль хранителя домашнего очага. Карлыгаш, переживающая измену мужа и стоящая перед непростым выбором. Молодая пара, мечтающая о ребенке, но столкнувшаяся с неразрешимой проблемой. Предстоит узнать, кто из них сохранит чувства, а кто решится начать всё сначала.",
      "genres": [
        {
          "name": "мелодрама"
        },
        {
          "name": "комедия"
        }
      ],
      "isSeries": false,
      "movieLength": 98,
      "name": "Свадьба — это 

In [43]:
import json
API_TOKEN = "V6FCA5R-JSG4X38-N4EB7Q2-FGCV21D"
HEADERS = {'X-API-KEY': API_TOKEN}
response = requests.get(
    "https://api.poiskkino.dev/v1.4/movie",
    headers=HEADERS,
    params={'limit': 1,  'selectFields': ['id', 'name', 'alternativeName', 'enName', 'year', 
                        'rating', 'description', 'genres', 'countries']}
)
if response.status_code == 200:
    data = response.json()
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    print(f"Ошибка {response.status_code}: {response.text}")

{
  "docs": [
    {
      "id": 11385598,
      "alternativeName": "Yйлену оңай",
      "countries": [
        {
          "name": "Казахстан"
        }
      ],
      "description": "Начинающий сценарист Айдар в поисках идеи для фильма об отношениях знакомится с психологом. Вместе они пытаются понять, что на самом деле происходит с любовью после свадьбы — и почему чувства, начинавшиеся как «навсегда», сталкиваются с реальностью. В их поле зрения — несколько историй. Успешный менеджер Жанар и её супруг Жандос, взявший на себя роль хранителя домашнего очага. Карлыгаш, переживающая измену мужа и стоящая перед непростым выбором. Молодая пара, мечтающая о ребенке, но столкнувшаяся с неразрешимой проблемой. Предстоит узнать, кто из них сохранит чувства, а кто решится начать всё сначала.",
      "genres": [
        {
          "name": "мелодрама"
        },
        {
          "name": "комедия"
        }
      ],
      "name": "Свадьба — это только начало",
      "rating": {
        "kp": 0,

In [44]:
from time import sleep
import random

API_TOKEN = "V6FCA5R-JSG4X38-N4EB7Q2-FGCV21D" 
BASE_URL = "https://api.poiskkino.dev"
HEADERS = {
    'X-API-KEY': API_TOKEN,
    'Content-Type': 'application/json'  
}
existing_titles = set(result_list['title']) if result_list['title'] else set()
target_count = 250
new_films_added = 0
page = 1
attempts = 0
max_attempts = 500  
while new_films_added < target_count and attempts < max_attempts:
    attempts += 1
    params = {
        'limit': 50, 
        'page': page,
        'selectFields': ['id', 'name', 'alternativeName', 'enName', 'year', 
                        'rating', 'description', 'genres', 'countries']
    }
    try:
        response = requests.get(
            f"{BASE_URL}/v1.4/movie", 
            headers=HEADERS, 
            params=params, 
            timeout=15
        )
        if response.status_code == 200:
            data = response.json()
            movies = data.get('docs', [])
            if not movies:
                print(f"На странице {page} нет фильмов.")
                break
            print(f"Получено {len(movies)} фильмов со страницы {page}.")
            for movie in movies:
                title_rus = movie.get('name') or movie.get('alternativeName')
                if not title_rus:
                    continue
                if title_rus in existing_titles:
                    continue
                title_eng = movie.get('enName') or movie.get('alternativeName') or "Н/Д"
                year = movie.get('year', "Н/Д")
                rating_obj = movie.get('rating', {})
                rating = rating_obj.get('kp')
                if rating:
                    rating_found = str(round(rating, 1))
                else:
                    rating_found = "Н/Д"
                description = movie.get('description') or movie.get('shortDescription') or "Н/Д"
                genres_list = movie.get('genres', [])
                if genres_list:
                    genres_names = [g.get('name', '') for g in genres_list if g.get('name')]
                    genres_found = ', '.join(genres_names) if genres_names else "Н/Д"
                else:
                    genres_found = "Н/Д"
                countries_list = movie.get('countries', [])
                if countries_list:
                    countries_names = [c.get('name', '') for c in countries_list if c.get('name')]
                    countries_found = ', '.join(countries_names) if countries_names else "Н/Д"
                else:
                    countries_found = "Н/Д"
                result_list['title'].append(title_rus)
                result_list['title_eng'].append(title_eng)
                result_list['genre'].append(genres_found)
                result_list['year'].append(str(year))
                result_list['country'].append(countries_found)
                result_list['rating'].append(rating_found)
                result_list['description'].append(description)
                existing_titles.add(title_rus)
                new_films_added += 1
                print(f"Добавлен {new_films_added}/{target_count}: {title_rus} ({year})")
                if new_films_added >= target_count:
                    break
            if data.get('page', 1) < data.get('pages', 1):
                page = data['page'] + 1
                print(f"  Переход на страницу {page}")
            else:
                print(f"Достигнут конец")
                break
        else:
            print(f"Ошибка API: {response.status_code}")
            if response.status_code == 401:
                print("Проблема с токеном!")
                break
            sleep(3)
    except requests.exceptions.Timeout:
        print(f"Таймаут запроса")
    except Exception as e:
        print(f"Ошибка: {e}")
    sleep(random.uniform(0.5, 1.5))
print(f"Добавлено {new_films_added} новых фильмов")
print(f"Всего в result_list теперь: {len(result_list['title'])} фильмов")

Получено 50 фильмов со страницы 1.
Добавлен 1/250: Свадьба — это только начало (2026)
Добавлен 2/250: The Curse of Highway Sheila (2014)
Добавлен 3/250: Abul eyoun garia (1958)
Добавлен 4/250: Remen (1978)
Добавлен 5/250: Амазонки русского авангарда (Н/Д)
Добавлен 6/250: Heroes of the Sand (2026)
Добавлен 7/250: Goat Girl (2026)
Добавлен 8/250: Untitled Jonestown/HBO Project (Н/Д)
Добавлен 9/250: Любовь И-сопа (Н/Д)
Добавлен 10/250: Skeletons (Н/Д)
Добавлен 11/250: Договорной брак (Н/Д)
Добавлен 12/250: Капитанская дочка (Н/Д)
Добавлен 13/250: Vasaris (2026)
Добавлен 14/250: Моя вторая удивительная жизнь (Н/Д)
Добавлен 15/250: Императрица из будущего (Н/Д)
Добавлен 16/250: Dynasty: The Murdochs (2026)
Добавлен 17/250: Доктор Щин (2026)
Добавлен 18/250: Disaster: The Chernobyl Meltdown (2026)
Добавлен 19/250: Марафон сердца (2026)
Добавлен 20/250: Les survivants: l'impossible départ apres la Shoah (2026)
Добавлен 21/250: Хоспис (Н/Д)
Добавлен 22/250: Талисман империи (2025)
Добавлен 23/

In [45]:
df = pd.DataFrame(result_list)
df.to_csv('top500_final.csv', index=False, encoding='utf-8-sig')
print(f"\nфильмов: {len(result_list['title'])}")
print("Файл сохранен:top500_final.csv")


фильмов: 500
Файл сохранен:top500_final.csv
